# Speckle-Denoised Sweep — NLM preprocessing + top-3 models + full evaluation

Same 3D-glaucoma sweep as before, but with **Non-Local Means (NLM) speckle denoising**
applied as preprocessing at 200³ (a standard, well-cited OCT despeckler).

**Pipeline:**
1. Stream Harvard-GF from HF → denoise every B-scan with **NLM** (`fast_mode`, h=0.1)
   → consolidated 200³ arrays cached on disk (per-volume **resume**, so a dropped
   Colab session never loses the ~3.6h precompute). Keep-alive cell included.
2. Show sample B-scans: **original vs denoised vs |diff|**.
3. **Sweep only the top-3 models** from the previous run (`enc-32-d5`, `enc-24`, `enc-32`).
   Every model logs live to **wandb** (project `glaucoma-thesis`) with full metrics
   (acc / precision / recall / f1) + test metrics.
4. **Full evaluation of the winner** (mirroring the 3DINO notebook):
   confusion matrix + report, **AUC-ROC + Sensitivity@Specificity**, **per-race fairness**,
   UMAP embeddings, and **3D Grad-CAM on false positives/negatives** (RNFL vs artifact check).

Run top to bottom on Colab **A100**. First run denoises + caches (~3.6h); re-runs are instant.


In [ ]:
!pip -q install monai umap-learn scikit-image wandb
!pip -q install hf-transfer huggingface_hub datasets


In [ ]:
from google.colab import drive, userdata
import os
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
if not os.environ.get("WANDB_API_KEY"):
    print("[wandb] WARNING: WANDB_API_KEY not set. Add secret WANDB_API_KEY=wandb_xxx")


In [ ]:
# ====== keep the Colab session alive during long runs ======
# Re-clicks Colab's "connect" button every 60s so an idle browser tab does not
# kill the runtime. Keep this tab open and unfocused is fine — but don't close it.
from google.colab import output

JS = """
setInterval(function(){
  const btn = document.querySelector("colab-connect-button");
  if (btn) btn.click();
}, 60000);
"""
try:
    output.eval_js(JS)
    print("[keepalive] armed — runtime will auto-reconnect while this tab stays open.")
except Exception as e:
    print("[keepalive] not available:", e)


In [ ]:
import os, io, json, time, zipfile
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from monai.networks import nets
from monai.networks.blocks import UnetBasicBlock

device = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True
_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if _bf16 else torch.float16
SEED = 42
RESOLUTION = 200


def to_device_normalize(x, y):
    x = x.to(device, non_blocking=True).float().div_(255.0)
    y = y.to(device, non_blocking=True)
    return x, y


# ====== wandb (mandatory) ======
def init_wandb(run_name, config=None):
    if not os.environ.get("WANDB_API_KEY"):
        return None
    import wandb
    try:
        return wandb.init(project="glaucoma-thesis", name=run_name,
                          config=config or {}, id=run_name, resume="allow")
    except Exception as e:
        print("[wandb] init failed, continuing without cloud logging:", e)
        return None


# ====== full metrics (acc / precision / recall / f1) ======
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


def score_row(y_true, y_pred):
    return dict(acc=float(accuracy_score(y_true, y_pred)),
                precision=float(precision_score(y_true, y_pred, zero_division=0)),
                recall=float(recall_score(y_true, y_pred, zero_division=0)),
                f1=float(f1_score(y_true, y_pred, zero_division=0)))


@torch.no_grad()
def eval_full(model, loader):
    model.eval(); ys, ps = [], []
    for x, y in loader:
        x, y = to_device_normalize(x, y)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            logits = model(x)
        ps.append(logits.argmax(1).cpu()); ys.append(y.cpu())
    return score_row(torch.cat(ys).numpy(), torch.cat(ps).numpy())


@torch.no_grad()
def predict_probs(model, loader):
    model.eval(); ps, ys = [], []
    for x, y in loader:
        x, y = to_device_normalize(x, y)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            logits = model(x)
        ps.append(torch.softmax(logits, 1)[:, 1].cpu()); ys.append(y.cpu())
    return torch.cat(ps).numpy(), torch.cat(ys).numpy()


In [ ]:
# ================== 200³ data with NLM speckle denoising (precomputed + cached) ==================
import concurrent.futures as cf
from skimage.restoration import denoise_nl_means

HF_REPO  = "harvardairobotics/Harvard-GF"
ZIP_FILE = "Dataset/dataset.zip"
CSV_FILE = "ReadMe/data_summary.csv"
DATA_DIR = "/content/glaucoma_hf_200_denoised"   # denoised .npy arrays (cached on disk)
SPLITS   = ("Training", "Validation", "Test")
BATCH_SIZE   = 2
GRAD_ACCUM   = 8
CACHE_IN_RAM = False
NUM_WORKERS  = max(2, os.cpu_count() or 2)
SPLIT_ALIAS  = {"training": "Training", "validation": "Validation", "valid": "Validation",
                "test": "Test", "testing": "Test"}

DENOISE = True
NLM_H = 0.1                        # NLM filter strength (noise std, on per-slice [0,1])
NLM_PATCH, NLM_DIST = 5, 3
NLM_WORKERS = min(8, os.cpu_count() or 2)

meta = {}
names = []
ZIP_PATH = None


def download_hf(filename):
    from huggingface_hub import hf_hub_download
    print(f"[data] downloading {HF_REPO}/{filename} ...", flush=True)
    return hf_hub_download(repo_id=HF_REPO, filename=filename, repo_type="dataset")


def ensure_meta():
    global meta, names, ZIP_PATH
    if meta:
        return
    csv_path = download_hf(CSV_FILE)
    ZIP_PATH = download_hf(ZIP_FILE)
    import csv
    with open(csv_path, newline="") as fh:
        for r in csv.DictReader(fh):
            split = SPLIT_ALIAS.get((r["use"] or "").strip().lower())
            if split is None:
                continue
            gl = 1 if str(r["glaucoma"]).strip().lower() in ("yes", "1", "true") else 0
            meta[Path(r["filename"]).stem] = (split, gl)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        names = [n for n in zf.namelist() if n.endswith(".npz")]
    print(f"[data] {len(meta)} labeled samples | {len(names)} zip entries")


def denoise_slice(sl):
    vmin, vmax = sl.min(), sl.max()
    if vmax - vmin < 1e-6:
        return sl
    sln = (sl.astype(np.float32) - vmin) / (vmax - vmin)
    den = denoise_nl_means(sln, h=NLM_H, patch_size=NLM_PATCH,
                           patch_distance=NLM_DIST, fast_mode=True)
    return np.clip(den * (vmax - vmin) + vmin, 0, 255).astype(np.uint8)


def denoise_volume(raw):
    """2D NLM on every B-scan of a (200,200,200) uint8 volume (slices parallel)."""
    out = np.empty_like(raw)
    slices = [raw[d] for d in range(raw.shape[0])]
    with cf.ThreadPoolExecutor(max_workers=NLM_WORKERS) as ex:
        for d, o in zip(range(raw.shape[0]), ex.map(denoise_slice, slices)):
            out[d] = o
    return out


def build_denoised_data():
    """Stream + denoise all volumes once, cache to DATA_DIR. Per-volume resume."""
    if all(os.path.isfile(os.path.join(DATA_DIR, f"{s}_volumes.npy")) for s in SPLITS):
        print(f"[data] denoised arrays already built at {DATA_DIR}")
        ensure_meta()
        return
    os.makedirs(DATA_DIR, exist_ok=True)
    ensure_meta()
    counts = {s: 0 for s in SPLITS}
    for n in names:
        m = meta.get(Path(n).stem)
        if m:
            counts[m[0]] += 1
    print("[data] zip-matched counts:", counts)

    exists = os.path.exists(os.path.join(DATA_DIR, "Training_volumes.npy"))
    mode = "r+" if exists else "w+"
    vols, labels = {}, {}
    for s in SPLITS:
        vols[s] = np.lib.format.open_memmap(os.path.join(DATA_DIR, f"{s}_volumes.npy"),
                                            mode=mode, dtype=np.uint8,
                                            shape=(counts[s], 1, 200, 200, 200))
        labels[s] = np.lib.format.open_memmap(os.path.join(DATA_DIR, f"{s}_labels.npy"),
                                              mode=mode, dtype=np.int64, shape=(counts[s],))

    progress_path = os.path.join(DATA_DIR, "progress.json")
    completed = json.load(open(progress_path)).get("stems", []) if os.path.exists(progress_path) else []
    done = set(completed)
    filled = {s: sum(1 for st in completed if meta[st][0] == s) for s in SPLITS}
    print(f"[data] resume: {len(completed)}/{len(names)} volumes already denoised")

    with zipfile.ZipFile(ZIP_PATH) as zf:
        for n in names:
            stem = Path(n).stem
            m = meta.get(stem)
            if not m:
                continue
            split, label = m
            if stem in done:
                continue
            raw = np.load(io.BytesIO(zf.read(n)))["oct_bscans"]          # (200,200,200) uint8
            if DENOISE:
                raw = denoise_volume(raw)
            vols[split][filled[split]] = raw[None]
            labels[split][filled[split]] = label
            filled[split] += 1
            completed.append(stem)
            if len(completed) % 25 == 0:
                for s in SPLITS:
                    vols[s].flush(); labels[s].flush()
                with open(progress_path, "w") as fh:
                    json.dump({"stems": completed}, fh)
                print(f"[data] {len(completed)}/{len(names)} volumes done", flush=True)
    for s in SPLITS:
        vols[s].flush(); labels[s].flush()
    with open(progress_path, "w") as fh:
        json.dump({"stems": completed}, fh)
    with open(os.path.join(DATA_DIR, "manifest.json"), "w") as fh:
        json.dump({"source": HF_REPO, "size_name": "200", "denoise": "NLM",
                   "nlm_h": NLM_H, "patch": NLM_PATCH, "dist": NLM_DIST,
                   "splits": {s: {"built_n": counts[s]} for s in SPLITS}}, fh, indent=2)
    print("[data] denoised arrays built at", DATA_DIR)


class OCTMemmapDataset(Dataset):
    def __init__(self, data_dir, split, cache_in_ram=False):
        self.labels = np.load(os.path.join(data_dir, f"{split}_labels.npy"))
        vp = os.path.join(data_dir, f"{split}_volumes.npy")
        self.volumes = np.load(vp) if cache_in_ram else np.load(vp, mmap_mode="r")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.from_numpy(np.ascontiguousarray(self.volumes[idx]).copy())
        y = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return x, y


def build_loaders():
    build_denoised_data()
    counts = {s: len(np.load(os.path.join(DATA_DIR, f"{s}_labels.npy"))) for s in SPLITS}
    print("[data] counts:", counts)
    workers = 0 if CACHE_IN_RAM else NUM_WORKERS
    kw = dict(batch_size=BATCH_SIZE, num_workers=workers, pin_memory=(device == "cuda"))
    if workers > 0:
        kw.update(persistent_workers=True, prefetch_factor=4)
    train_ds = OCTMemmapDataset(DATA_DIR, "Training",   cache_in_ram=CACHE_IN_RAM)
    val_ds   = OCTMemmapDataset(DATA_DIR, "Validation", cache_in_ram=CACHE_IN_RAM)
    test_ds  = OCTMemmapDataset(DATA_DIR, "Test",       cache_in_ram=CACHE_IN_RAM)
    g = torch.Generator(); g.manual_seed(SEED)
    train_loader = DataLoader(train_ds, shuffle=True, generator=g, **kw)
    val_loader   = DataLoader(val_ds,   shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  shuffle=False, **kw)
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = build_loaders()


In [ ]:
# ---- conservative 3D augmentation (training only) ----
from monai.transforms import (Compose, RandFlip, RandRotate, RandScaleIntensity,
                              RandShiftIntensity, RandGaussianNoise)


def make_train_transform():
    return Compose([
        RandFlip(prob=0.5, spatial_axis=1),
        RandFlip(prob=0.5, spatial_axis=2),
        RandRotate(range_x=0.10, range_y=0.10, range_z=0.10,
                   prob=0.5, mode="bilinear", padding_mode="zeros", keep_size=True),
        RandScaleIntensity(factors=0.10, prob=0.5),
        RandShiftIntensity(offsets=10.0, prob=0.5),
        RandGaussianNoise(prob=0.3, std=5.0),
    ])


class TrainAug:
    def __init__(self, ds, tf):
        self.ds, self.tf = ds, tf
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        x, y = self.ds[i]
        x = torch.as_tensor(self.tf(x)[0])
        if x.ndim == 3:
            x = x.unsqueeze(0)
        return x, y


g = torch.Generator(); g.manual_seed(SEED)
workers = 0 if CACHE_IN_RAM else NUM_WORKERS
train_loader = DataLoader(TrainAug(train_loader.dataset, make_train_transform()),
                          batch_size=BATCH_SIZE, shuffle=True, generator=g,
                          num_workers=workers, pin_memory=(device == "cuda"),
                          persistent_workers=workers > 0, prefetch_factor=4 if workers > 0 else None)


In [ ]:
# ====== Sample images: original vs NLM-denoised B-scans ======
import matplotlib.pyplot as plt

SAMPLE_DIR = os.path.join(DATA_DIR, "samples")
os.makedirs(SAMPLE_DIR, exist_ok=True)

test_order = [Path(n).stem for n in names if meta.get(Path(n).stem, ("", ""))[0] == "Test"]
entries = {Path(n).stem: n for n in names}
with zipfile.ZipFile(ZIP_PATH) as zf:
    for stem in test_order[:3]:
        raw = np.load(io.BytesIO(zf.read(entries[stem])))["oct_bscans"]        # original
        row = test_order.index(stem)
        den = np.asarray(test_loader.dataset.volumes[row])[0]                 # (200,200,200) denoised
        mid = 100
        diff = np.abs(raw[mid].astype(np.float32) - den[mid].astype(np.float32))
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        for ax, im, t in zip(axes, (raw[mid], den[mid], diff),
                             ("original", "NLM-denoised", "|diff|")):
            ax.imshow(im, cmap="gray")
            ax.set_title(f"{t} (B-scan {mid})")
            ax.axis("off")
        fig.suptitle(f"sample {stem} | NLM h={NLM_H} patch={NLM_PATCH} dist={NLM_DIST}")
        fig.savefig(os.path.join(SAMPLE_DIR, f"{stem}.png"), dpi=120, bbox_inches="tight")
        plt.show()
print(f"[samples] saved -> {SAMPLE_DIR}")


In [ ]:
class UNetEncoder3D(nn.Module):
    """MONAI UNet ENCODER ONLY (decoder removed) + AdaptiveAvgPool + classification head."""

    def __init__(self, in_channels=1, num_classes=2, features=(32, 64, 128, 256),
                 strides=(2, 2, 2), num_res_units=2, norm="batch", dropout=0.0):
        super().__init__()
        self.features = tuple(features)
        strides = tuple(strides)
        if len(strides) != len(features) - 1:
            strides = (2,) * (len(features) - 1)
        self.down_blocks = nn.ModuleList()
        self.down_samples = nn.ModuleList()
        cin = in_channels
        for i, f in enumerate(features):
            self.down_blocks.append(UnetBasicBlock(
                3, cin, f, kernel_size=3, stride=1,
                norm_name=norm, act_name="relu", dropout=dropout))
            cin = f
            if i < len(features) - 1:
                self.down_samples.append(nn.MaxPool3d(strides[i], stride=strides[i]))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(features[-1], num_classes),
        )

    def forward(self, x):
        for blk, samp in zip(self.down_blocks, self.down_samples + [nn.Identity()]):
            x = blk(x)
            x = samp(x)
        return self.head(x)


class Simple3DCNN(nn.Module):
    """Project reference baseline (models/glaucoma/model.py, unchanged)."""

    def __init__(self, in_channels=1, num_classes=2, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(in_channels, 16, kernel_size=3, padding=1), nn.BatchNorm3d(16),
            nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(16, 32, kernel_size=3, padding=1), nn.BatchNorm3d(32),
            nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(32, 64, kernel_size=3, padding=1), nn.BatchNorm3d(64),
            nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(64, 128, kernel_size=3, padding=1), nn.BatchNorm3d(128),
            nn.ReLU(), nn.AdaptiveAvgPool3d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def build_model(spec):
    if "features" in spec:
        return UNetEncoder3D(in_channels=1, num_classes=2,
                             features=spec["features"],
                             num_res_units=spec.get("num_res_units", 2))
    return Simple3DCNN(in_channels=1, num_classes=2)


In [ ]:
# ====== sweep: top-3 models from the previous run, on NLM-denoised data (wandb + resume) ======
SWEEP = [
    {"family": "MONAI-UNetEncoder", "name": "enc-32-d5", "features": (32, 64, 128, 256, 512), "num_res_units": 2},
    {"family": "MONAI-UNetEncoder", "name": "enc-24",     "features": (24, 48, 96, 192),     "num_res_units": 2},
    {"family": "MONAI-UNetEncoder", "name": "enc-32",     "features": (32, 64, 128, 256),    "num_res_units": 2},
]
SWEEP_EPOCHS = 20
EFFECTIVE_BS = BATCH_SIZE * GRAD_ACCUM
LR  = 1e-4 * (EFFECTIVE_BS / 4) ** 0.5
WD, PATIENCE = 1e-4, 10
SAVE_DIR = "/content/drive/MyDrive/MasterBKDN/Thesis/denoise_sweep"
os.makedirs(SAVE_DIR, exist_ok=True)


def _probe_loader(bs):
    return DataLoader(train_loader.dataset, batch_size=bs, shuffle=True,
                      num_workers=0, generator=torch.Generator().manual_seed(SEED))


def static_probe(spec, bs=None):
    bs = BATCH_SIZE if bs is None else bs
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    try:
        m = build_model(spec).to(device)
        opt = torch.optim.AdamW(m.parameters(), LR, weight_decay=WD)
        xb, yb = next(iter(_probe_loader(bs))); xb, yb = to_device_normalize(xb, yb)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            loss = nn.functional.cross_entropy(m(xb), yb)
        loss.backward(); opt.step()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        params = sum(p.numel() for p in m.parameters()) / 1e6
        vram = (torch.cuda.max_memory_allocated() / 1e9) if torch.cuda.is_available() else 0.0
        del m, opt, xb, yb, loss; torch.cuda.empty_cache()
        return params, vram, ("ok" if bs == BATCH_SIZE else "ok@bs1"), bs
    except Exception as e:
        torch.cuda.empty_cache()
        if bs > 1:
            print(f"  [probe] bs={bs} failed ({type(e).__name__}: {str(e)[:160]}); retrying bs=1")
            return static_probe(spec, bs=1)
        st = "OOM" if "out of memory" in str(e).lower() else "ERR"
        print(f"  [probe] {st}: {type(e).__name__}: {str(e)[:300]}")
        return float("nan"), float("nan"), st, 1


def train_one(spec, bs=BATCH_SIZE):
    torch.manual_seed(SEED); torch.cuda.empty_cache()
    name = spec["name"]
    wb = init_wandb(f"denoise-{name}", config={"model": name, "preprocess": "NLM",
                                               "resolution": RESOLUTION, "epochs": SWEEP_EPOCHS,
                                               "lr": LR, "wd": WD, "bs": bs})
    try:
        model = build_model(spec).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=SWEEP_EPOCHS)
        scaler = torch.amp.GradScaler("cuda",
                    enabled=(USE_AMP and device == "cuda" and amp_dtype == torch.float16))
        crit = nn.CrossEntropyLoss()
        loader = train_loader if bs == BATCH_SIZE else _probe_loader(bs)
        best_val = 0.0; best_test = None; best_ep = -1; bad = 0; ep_times = []
        for ep in range(SWEEP_EPOCHS):
            model.train(); t0 = time.time(); opt.zero_grad(set_to_none=True)
            for i, (x, y) in enumerate(loader):
                x, y = to_device_normalize(x, y)
                with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
                    loss = crit(model(x), y) / GRAD_ACCUM
                scaler.scale(loss).backward()
                if (i + 1) % GRAD_ACCUM == 0:
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            sched.step(); ep_times.append(time.time() - t0)
            vm = eval_full(model, val_loader)
            if wb is not None:
                wb.log({"epoch": ep + 1, **{f"val/{k}": v for k, v in vm.items()},
                        "val/best_acc": best_val}, step=ep + 1)
            if vm["acc"] > best_val:
                best_val, best_ep, bad = vm["acc"], ep, 0
                best_test = eval_full(model, test_loader)
                torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"best_{name}.pt"))
            else:
                bad += 1
                if bad >= PATIENCE:
                    break
            print(f"  [{name}] ep{ep+1:02d} val=" + " ".join(f"{k}={v:.4f}" for k, v in vm.items())
                  + f" (best {best_val:.4f})", end="\r")
        del model, opt; torch.cuda.empty_cache()
        if wb is not None:
            wb.summary.update({"best_val": best_val, "best_ep": best_ep + 1, "test": best_test or {}})
            wb.finish()
        return dict(val=best_val, test=(best_test or {}).get("acc", float("nan")),
                    test_metrics=best_test, best_ep=best_ep + 1,
                    sec_ep=sum(ep_times) / len(ep_times), status="ok", ckpt=f"best_{name}.pt")
    except RuntimeError as e:
        torch.cuda.empty_cache()
        if wb is not None:
            wb.finish()
        st = "OOM" if "out of memory" in str(e).lower() else "ERR"
        print(f"  [train] {st}: {type(e).__name__}: {str(e)[:300]}")
        return dict(val=float("nan"), test=float("nan"), best_ep=-1,
                    sec_ep=float("nan"), status=st)


# ====== run (resume-safe, saves after each model) ======
RESULTS_LOCAL = "/content/sweep_results_denoise.json"
RESULTS_DRIVE = "/content/drive/MyDrive/MasterBKDN/Thesis/sweep_results_denoise.json"


def load_existing():
    for p in (RESULTS_LOCAL, RESULTS_DRIVE):
        try:
            with open(p) as fh:
                return json.load(fh)
        except Exception:
            continue
    return []


def save_rows(rows):
    try:
        with open(RESULTS_LOCAL, "w") as fh:
            json.dump(rows, fh, indent=2)
        os.makedirs(os.path.dirname(RESULTS_DRIVE), exist_ok=True)
        import shutil
        shutil.copy(RESULTS_LOCAL, RESULTS_DRIVE)
        print(f"  [save] {len(rows)} results -> {RESULTS_DRIVE}", flush=True)
    except Exception as e:
        print("  [save] skipped:", e, flush=True)


rows = load_existing()
done = {r["name"] for r in rows}
for spec in SWEEP:
    name = spec["name"]
    print(f"\n=== [{spec['family']}] {name} (NLM-denoised) ===")
    if name in done:
        print("  already done -> skipping (resume); see row below")
        continue
    params, vram, st, bs = static_probe(spec)
    if st not in ("ok", "ok@bs1"):
        print(f"  static probe: {st}")
        rows.append(dict(family=spec["family"], name=name, params=params, vram=vram,
                         val=float("nan"), test=float("nan"), best_ep=-1,
                         sec_ep=float("nan"), status=st, bs=bs,
                         features=spec.get("features"), num_res_units=spec.get("num_res_units")))
        save_rows(rows)
        continue
    print(f"  params {params:.1f}M | VRAM {vram:.1f}GB | batch={bs}")
    r = train_one(spec, bs=bs)
    r.update(family=spec["family"], name=name, params=params, vram=vram, bs=bs,
             features=spec.get("features"), num_res_units=spec.get("num_res_units"))
    rows.append(r)
    save_rows(rows)
    print(f"  val={r['val']:.4f} test={r['test']:.4f} @ep{r['best_ep']} | {r['sec_ep']:.1f}s/ep   ")


def fmt(r):
    return (f"{r['name']:18s} {r['params']:>6.1f}M {r['vram']:>5.1f}G {r['sec_ep']:>5.1f}s "
            f"{r['val']:>7.4f} {r['test']:>7.4f} {r['best_ep']:>6d}  b{r.get('bs', BATCH_SIZE)}  {r['status']}")

hdr = f"{'model':18s} {'params':>7} {'VRAM':>6} {'s/ep':>6} {'Val':>7} {'Test':>7} {'bestEp':>6} {'bs':>2}"
print("\n============= SWEEP 200³ NLM-DENOISED (top-3) =============")
for r in rows:
    print(fmt(r))
ok = sorted([r for r in rows if r["status"] == "ok"], key=lambda r: -r["val"])
print("\n----- TOP theo Val -----")
for r in ok:
    print(fmt(r))
if ok:
    WINNER = ok[0]
    print(f"\n>> WINNER: {WINNER['name']} Val={WINNER['val']:.4f} Test={WINNER['test']:.4f} "
          f"| test_metrics={WINNER.get('test_metrics')}")
else:
    WINNER = None


In [ ]:
# ====== Evaluation on the WINNER of the denoised sweep ======
assert WINNER is not None, "run the sweep cell first"
print(f"[eval] winner: {WINNER['name']} | val={WINNER['val']:.4f} test={WINNER['test']:.4f}")
wmodel = build_model(WINNER).to(device)
wmodel.load_state_dict(torch.load(os.path.join(SAVE_DIR, WINNER["ckpt"]), map_location=device))
wmodel.eval()

# ---- confusion matrix + classification report (full metrics on test) ----
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

probs, ytest = predict_probs(wmodel, test_loader)
pred = (probs >= 0.5).astype(int)
fair_metrics = score_row(ytest, pred)
print("[eval] test " + " | ".join(f"{k}={v:.4f}" for k, v in fair_metrics.items()))
print(classification_report(ytest, pred, target_names=["no_glaucoma", "glaucoma"]))
print("confusion matrix:\n", confusion_matrix(ytest, pred))

import matplotlib.pyplot as plt
ConfusionMatrixDisplay(confusion_matrix(ytest, pred), display_labels=["no_glaucoma", "glaucoma"]).plot(cmap="Blues")
plt.title(f"{WINNER['name']} confusion matrix (NLM-denoised)")
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ====== AUC-ROC + Sensitivity @ Specificity (winner) ======
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.calibration import calibration_curve

fair_auc = roc_auc_score(ytest, probs)
fair_brier = brier_score_loss(ytest, probs)


def sens_at_spec(y_true, probs, specs=(0.90, 0.95, 0.99)):
    """Max sensitivity while keeping specificity >= each target."""
    fpr, tpr, _ = roc_curve(y_true, probs)
    out = {}
    for sp in specs:
        j = np.where((1 - fpr) >= sp)[0]
        out[f"sens@{int(sp * 100)}"] = float(tpr[j[-1]]) if len(j) else float("nan")
    return out


sens_spec = sens_at_spec(ytest, probs)
print(f"[eval] AUC-ROC = {fair_auc:.4f}")
print("[eval] Sensitivity @ Specificity:", sens_spec)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fpr, tpr, _ = roc_curve(ytest, probs)
axes[0].plot(fpr, tpr, label=f"{WINNER['name']} (AUC={fair_auc:.3f})")
for sp in (0.90, 0.95, 0.99):
    j = np.where((1 - fpr) >= sp)[0]
    if len(j):
        idx = j[-1]
        axes[0].scatter(fpr[idx], tpr[idx], marker="o", s=45,
                        label=f"spec={sp:.0%} sens={tpr[idx]:.2f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("ROC (test)"); axes[0].legend()
pt, pp = calibration_curve(ytest, probs, n_bins=10)
axes[1].plot(pp, pt, marker="o", label=f"Brier={fair_brier:.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_xlabel("predicted prob"); axes[1].set_ylabel("fraction positive"); axes[1].set_title("Calibration"); axes[1].legend()
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "roc_calibration.png"), dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ====== Fairness: per-race accuracy / AUC on test (Harvard-GF is a fairness dataset) ======
import zipfile, csv as _csv
import numpy as np
from sklearn.metrics import roc_auc_score

race_code = {"asian": 0, "black": 1, "white": 2}

def ensure_meta():  # already set by the data cell
    pass

csv_path = download_hf(CSV_FILE)
rc_meta = {}
with open(csv_path, newline="") as fh:
    for r in _csv.DictReader(fh):
        split = SPLIT_ALIAS.get((r["use"] or "").strip().lower())
        if split is None:
            continue
        rc = race_code.get(str(r["race"]).strip().lower(), 3)
        rc_meta[Path(r["filename"]).stem] = (split, rc)
test_races = [rc_meta[Path(n).stem][1] for n in names
              if Path(n).stem in rc_meta and rc_meta[Path(n).stem][0] == "Test"]
assert len(test_races) == len(test_loader.dataset), (len(test_races), len(test_loader.dataset))
races = np.array(test_races)

per_race = {}
print("[fairness] race distribution:", {n: int((races == c).sum()) for n, c in
                                        (("asian", 0), ("black", 1), ("white", 2), ("other", 3))})
for code, name in ((0, "asian"), (1, "black"), (2, "white"), (3, "other")):
    m = races == code
    if m.sum() == 0:
        continue
    acc = (pred[m] == ytest[m]).mean()
    auc = roc_auc_score(ytest[m], probs[m]) if len(np.unique(ytest[m])) > 1 else float("nan")
    per_race[name] = {"n": int(m.sum()), "acc": float(acc), "auc": float(auc)}
    print(f"  race={name:6s} n={m.sum():4d} acc={acc:.4f} auc={auc:.4f}")


In [ ]:
# ====== UMAP of penultimate embeddings (winner) — class separability ======
import numpy as np
import matplotlib.pyplot as plt


def penultimate(model, x):
    if hasattr(model, "head"):
        target = model.head[0]
    else:
        target = [m for m in model.features.modules() if isinstance(m, nn.AdaptiveAvgPool3d)][-1]
    pooled = {}
    handle = target.register_forward_hook(lambda m, i, o: pooled.__setitem__("e", o.detach().flatten(1)))
    x = x.float().div_(255.0).to(device)          # uint8 -> [0,1] (model expects float)
    with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
        model(x)
    handle.remove()
    return pooled["e"]


@torch.no_grad()
def collect_embeddings(model, ds, max_n=500, seed=SEED):
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(ds), size=min(max_n, len(ds)), replace=False)
    Es, Ys = [], []
    for i in idxs:
        x, y = ds[int(i)]
        Es.append(penultimate(model, x.unsqueeze(0)).cpu())
        Ys.append(int(y))
    return torch.cat(Es).numpy(), np.array(Ys)


from umap import UMAP
E, Y = collect_embeddings(wmodel, test_loader.dataset, max_n=500)
if E.shape[0] < 3:
    print(f"[umap] only {E.shape[0]} samples, skipping plot")
else:
    n_nb = min(15, max(2, E.shape[0] - 1))
    emb = UMAP(n_neighbors=n_nb, min_dist=0.1, random_state=SEED).fit_transform(E)
    fig, ax = plt.subplots(figsize=(7, 6))
    for c, lab, col in ((0, "no_glaucoma", "tab:blue"), (1, "glaucoma", "tab:red")):
        m = Y == c
        ax.scatter(emb[m, 0], emb[m, 1], s=12, c=col, label=f"{lab} (n={m.sum()})", alpha=0.7)
    ax.legend(); ax.set_title(f"{WINNER['name']} UMAP (NLM-denoised, test 500)")
    fig.savefig(os.path.join(SAVE_DIR, "umap_embeddings.png"), dpi=120, bbox_inches="tight")
    plt.show()


In [ ]:
# ====== 3D Grad-CAM on False Positives / False Negatives (RNFL vs artifact check) ======
import numpy as np
import matplotlib.pyplot as plt


def _target_block(model):
    if hasattr(model, "down_blocks"):
        return model.down_blocks[-1]
    return [m for m in model.features.modules() if isinstance(m, nn.Conv3d)][-1]


def grad_cam(model, x, target=None):
    model.eval()
    if target is None:
        target = _target_block(model)
    store = {}
    handle = target.register_forward_hook(lambda m, i, o: store.__setitem__("a", o))
    with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
        out = model(x)
    handle.remove()
    cls = int(out.argmax(1))
    score = out[0, cls]
    a = store["a"]
    grad = torch.autograd.grad(score, a)[0]
    w = grad.mean(dim=(2, 3, 4), keepdim=True)
    cam = (w * a).sum(1, keepdim=True).relu()
    cam = F.interpolate(cam, size=x.shape[2:], mode="trilinear", align_corners=False)
    cam = cam.squeeze(0, 1).detach().cpu()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam, cls, torch.softmax(out, 1)[0, cls].item()


def border_mass(cam, margin=0.1):
    D, H, W = cam.shape
    m = int(D * margin)
    border = torch.zeros_like(cam, dtype=torch.bool)
    border[:m] = border[-m:] = True
    border[:, :m] = border[:, -m:] = True
    border[:, :, :m] = border[:, :, -m:] = True
    return float(cam[border].sum() / (cam.sum() + 1e-8))


def load_sample(ds, idx):
    x, y = ds[idx]
    return x.float().div_(255.0).unsqueeze(0).to(device), int(y)


def show_overlay(vol, cam, title, fname, axis=0, fracs=(0.4, 0.5, 0.6)):
    fig, axes = plt.subplots(1, len(fracs), figsize=(5 * len(fracs), 5))
    for ax, fr in zip(axes, fracs):
        idx = int(vol.shape[axis] * fr)
        sl = vol[idx] if axis == 0 else vol[:, idx, :] if axis == 1 else vol[:, :, idx]
        cm = cam[idx] if axis == 0 else cam[:, idx, :] if axis == 1 else cam[:, :, idx]
        ax.imshow(sl, cmap="gray")
        ax.imshow(cm, cmap="jet", alpha=0.5, vmin=0.0, vmax=1.0)
        ax.set_title(f"sl={idx}")
        ax.axis("off")
    fig.suptitle(title)
    fig.savefig(os.path.join(SAVE_DIR, fname), dpi=120, bbox_inches="tight")
    plt.show()


fp_idx = np.where((pred == 1) & (ytest == 0))[0]
fn_idx = np.where((pred == 0) & (ytest == 1))[0]
print(f"[gradcam] test FP={len(fp_idx)} | FN={len(fn_idx)}")

for kind, group in (("FP", fp_idx), ("FN", fn_idx)):
    for j in group[:3]:
        x, y = load_sample(test_loader.dataset, int(j))
        cam, cls, p = grad_cam(wmodel, x)
        bm = border_mass(cam)
        hint = ("WARNING: saliency near borders -> possible background/artifact shortcut"
                if bm > 0.5 else "saliency mostly central -> inspect if it lands on RNFL / neuroretinal rim")
        print(f"{kind} idx={int(j)} true={y} pred={cls} p={p:.3f} | border_mass={bm:.2f} -> {hint}")
        vol = x.squeeze(0, 1).cpu().numpy()
        for axis, aname in ((0, "axial"), (1, "sagittal"), (2, "coronal")):
            show_overlay(vol, cam.numpy(), f"Grad-CAM {kind} true={y} pred={cls} ({aname})",
                         f"gradcam_{kind}_{int(j)}_{aname}.png", axis=axis)


In [ ]:
# ====== summary + save to Drive ======
g = globals()


def _get(name, default=None):
    return g.get(name, default)


summary = {
    "experiment": "SOTA-style preprocessing comparison — NLM speckle denoising @ 200³",
    "preprocess": {"method": "NLM", "h": NLM_H, "patch_size": NLM_PATCH,
                   "patch_distance": NLM_DIST, "resolution": RESOLUTION},
    "winner": {"name": _get("WINNER") and _get("WINNER")["name"],
               "val": _get("WINNER") and _get("WINNER")["val"],
               "test": _get("WINNER") and _get("WINNER")["test"]},
    "test_metrics": _get("fair_metrics"),
    "auc": _get("fair_auc"), "brier": _get("fair_brier"),
    "sens_at_spec": _get("sens_spec"),
    "fairness_per_race": _get("per_race"),
    "previous_no_denoise_best": {"val": 0.7967, "test": 0.7533, "model": "enc-32-d5"},
}
print(json.dumps(summary, indent=2))
try:
    out = "/content/drive/MyDrive/MasterBKDN/Thesis/denoise_sweep_results.json"
    with open(out, "w") as fh:
        json.dump(summary, fh, indent=2)
    print(f"[saved] denoise_sweep_results.json -> {out}")
except Exception as e:
    print("save skipped:", e)


In [ ]:
# Done. Release the GPU immediately.
from google.colab import runtime
runtime.unassign()
